In [1]:
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING


2026-03-09 23:07:34.720017: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 23:07:34.746480: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-09 23:07:35.165616: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress th

# FUNCTIONS FOR CALCULATING EFFICIENCY

In [2]:

def bernstein_1d(n, k, t):
    """Bernstein base polynomial B_{n,k}(t) on t in [0,1]."""
    return comb(n, k) * (t**k) * ((1.0 - t)**(n - k))


def bernstein2d_matrix(nx, ny, x, y):
    """
    Build the design matrix for Bernstein2D basis.
    Input: x, y in [-1,1] (arrays)
    Output: B matrix of size (Npoints, (nx+1)*(ny+1))
    """
    # map [-1,1] -> [0,1]
    tx = 0.5*(x + 1.0)
    ty = 0.5*(y + 1.0)

    B_list = []
    for i in range(nx+1):
        for j in range(ny+1):
            B_list.append(bernstein_1d(nx, i, tx) * bernstein_1d(ny, j, ty))
    B = np.vstack(B_list).T   # shape (Npoints, Ncoef)
    return B


# ======================================================
# 2) Fit Bernstein2D to a 2D efficiency map
# ======================================================
def fit_bernstein2d( xcenters, ycenters, eff2d, ngen2d, nx=8, ny=8, min_counts_mask=None,reg_lambda=1e-10,):
    """
    Fits a Bernstein2D polynomial to a 2D efficiency map using least squares + Tikhonov reg.

    Inputs:
        xcenters, ycenters   1D arrays (bin centers)
        eff2d                2D array with efficiency values (NaNs allowed)
        nx, ny               polynomial orders
        min_counts_mask      Boolean 2D mask (valid bins: True)
        reg_lambda           regularization parameter

    Returns:
        coef                 fitted coefficients
        eff_model            modeled efficiency on the grid

    
    Weighted fit of a Bernstein2D polynomial to a 2D efficiency map.

    The weights are derived from binomial uncertainties:
        sigma^2 = eff * (1 - eff) / ngen
    """

    XX, YY = np.meshgrid(xcenters, ycenters, indexing="ij")
    xflat = XX.ravel()
    yflat = YY.ravel()
    eff_flat = eff2d.ravel()
    ngen_flat = ngen2d.ravel()

    # Valid bins
    if min_counts_mask is None: use = (~np.isnan(eff_flat)) & (ngen_flat > 0)
    else:
        use = (min_counts_mask.ravel() & ~np.isnan(eff_flat)& (ngen_flat > 0))

    x_use = xflat[use]
    y_use = yflat[use]
    eff_use = eff_flat[use]
    ngen_use = ngen_flat[use]

    # Binomial uncertainty
    sigma2 = eff_use * (1.0 - eff_use) / ngen_use
    sigma2 = np.clip(sigma2, 1e-12, None)
    w = 1.0 / np.sqrt(sigma2)
    B = bernstein2d_matrix(nx, ny, x_use, y_use)

    # Apply weights
    Bw = B * w[:, None]
    yw = eff_use * w

    # Regularized weighted least squares
    BTB = Bw.T @ Bw + reg_lambda * np.eye(B.shape[1])
    BTy = Bw.T @ yw
    coef = np.linalg.solve(BTB, BTy)
    Bfull = bernstein2d_matrix(nx, ny, xflat, yflat)
    eff_model_flat = Bfull @ coef
    eff_model = eff_model_flat.reshape(eff2d.shape)
    return coef, eff_model


def build_efficiency_2d(gen_all_x, gen_all_y, gen_fid_x, gen_fid_y, reco_fid_x, reco_fid_y, reco_x, reco_y, 
                        weights_reco=None, nbx=20, nby=20, nxg=8, nyg=8, nxr=8, nyr=8, min_gen=0, reg_acc=1e-4, reg_reco=1e-4):
    xedges = np.linspace(-1, 1, nbx + 1)
    yedges = np.linspace(-1, 1, nby + 1)
    xcenters = 0.5 * (xedges[:-1] + xedges[1:])
    ycenters = 0.5 * (yedges[:-1] + yedges[1:])

    # Pesos 
    if weights_reco is None:
        weights_reco = np.ones(len(reco_x))

    # Histograms
    gen_allH, _, _ = np.histogram2d(gen_all_x, gen_all_y, bins=[xedges, yedges])
    gen_fidH, _, _ = np.histogram2d(gen_fid_x, gen_fid_y, bins=[xedges, yedges])
    # Denominador Reco
    reco_fidH, _, _ = np.histogram2d(reco_fid_x, reco_fid_y, bins=[xedges, yedges])    
    # Numerador Reco (CON PESOS APLICADOS)
    recoH, _, _ = np.histogram2d(reco_x, reco_y, bins=[xedges, yedges], weights=weights_reco)

    # ============================
    # Acceptance
    # ============================
    mask_gen = gen_allH > min_gen
    acc_gen = np.full_like(gen_allH, np.nan)
    with np.errstate(divide='ignore', invalid='ignore'):
        acc_gen[mask_gen] = gen_fidH[mask_gen] / gen_allH[mask_gen]

    # ============================
    # Reconstruction efficiency
    # ============================
    eff_reco = np.full_like(gen_allH, np.nan)
    valid = mask_gen & (reco_fidH > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        eff_reco[valid] = recoH[valid] / reco_fidH[valid]
    
    # ============================
    # Bernstein fits
    # ============================
    # Nota: Los fits también deben saber que recoH ahora tiene pesos (float), no solo counts (int).
    # Asegúrate de que fit_bernstein2d maneje arrays de floats en "data_hist" (recoH).
    coef_acc, acc_gen_model = fit_bernstein2d( xcenters, ycenters, acc_gen, gen_allH, nx=nxg, ny=nyg, min_counts_mask=mask_gen, reg_lambda=reg_acc)
    coef_reco, eff_reco_model = fit_bernstein2d(xcenters, ycenters, eff_reco, reco_fidH, nx=nxr, ny=nyr, min_counts_mask=valid, reg_lambda=reg_reco)

    return (xcenters, ycenters, acc_gen, acc_gen_model, coef_acc, eff_reco, eff_reco_model, coef_reco, mask_gen)


def build_efficiency_1d(gen_all, gen_fid, reco_fid, reco, weights_reco=None, nbins=30, n_poly=4, min_gen=10, reg_acc=1e-6, reg_reco=1e-5):

    limit_min, limit_max = -np.pi, np.pi
    edges = np.linspace(limit_min, limit_max, nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    centers_norm = 2 * (centers - limit_min) / (limit_max - limit_min) - 1.0

    # Pesos
    if weights_reco is None:
        weights_reco = np.ones(len(reco))
    
    # Histograms
    gen_allH, _ = np.histogram(gen_all, bins=edges)
    gen_fidH, _ = np.histogram(gen_fid, bins=edges)
    reco_fidH, _ = np.histogram(reco_fid, bins=edges)
    
    # Numerador Reco CON PESOS
    recoH, _ = np.histogram(reco, bins=edges, weights=weights_reco)

    # Acceptance
    mask_gen = gen_allH > min_gen
    acc_gen = np.full_like(gen_allH, np.nan, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        acc_gen[mask_gen] = gen_fidH[mask_gen] / gen_allH[mask_gen]
    coef_acc, acc_model = fit_bernstein1d( centers_norm, acc_gen, gen_allH, n=n_poly, min_counts_mask=mask_gen, reg_lambda=reg_acc)
    
    # Efficiency reco
    eff_reco = np.full_like(gen_allH, np.nan, dtype=float)
    valid_reco = mask_gen & (reco_fidH > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        eff_reco[valid_reco] = recoH[valid_reco] / reco_fidH[valid_reco]
    coef_reco, eff_reco_model = fit_bernstein1d(centers_norm, eff_reco, reco_fidH, n=n_poly, min_counts_mask=valid_reco, reg_lambda=reg_reco)

    return centers, acc_gen, acc_model, coef_acc, eff_reco, eff_reco_model, coef_reco, mask_gen


def bernstein2d_eval(x, y, model):
    """
    Evaluate a fitted Bernstein2D model.
    """
    nx = model["nx"]
    ny = model["ny"]
    coef = np.asarray(model["coef"])
    tx = 0.5 * (x + 1.0)
    ty = 0.5 * (y + 1.0)
    eff = np.zeros_like(tx, dtype=float)
    idx = 0
    for i in range(nx + 1):
        Bx = bernstein_1d(nx, i, tx)
        for j in range(ny + 1):
            By = bernstein_1d(ny, j, ty)
            eff += coef[idx] * Bx * By
            idx += 1

    return eff


# ======================================================
# Save / Load Bernstein models
# ======================================================
def save_bernstein2d_model(filename, coef, nx, ny):
    model = {"nx": nx, "ny": ny, "coef": coef.tolist(), "x_range": [-1.0, 1.0], "y_range": [-1.0, 1.0]}
    directory = os.path.dirname(filename)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(filename, "w") as f:
        json.dump(model, f, indent=2)


def load_bernstein_model(filename):
    with open(filename) as f:
        model = json.load(f)
    return (np.asarray(model["coef"], dtype=np.float64), model["nx"], model["ny"])


# ======================================================
#   plots
# ======================================================
def project_with_errors_x(data2d, mask):
    """Project 2D efficiency to x with diagnostic errors."""
    proj = []
    err = []
    for i in range(data2d.shape[0]):   # x bins
        vals = data2d[i, :][mask[i, :]]
        if len(vals) == 0:
            proj.append(np.nan)
            err.append(np.nan)
        else:
            proj.append(np.mean(vals))
            err.append(np.std(vals, ddof=0) / np.sqrt(len(vals)))
    return np.array(proj), np.array(err)


def project_with_errors_y(data2d, mask):
    """Project 2D efficiency to y with diagnostic errors."""
    proj = []
    err = []
    for j in range(data2d.shape[1]):   # y bins
        vals = data2d[:, j][mask[:, j]]
        if len(vals) == 0:
            proj.append(np.nan)
            err.append(np.nan)
        else:
            proj.append(np.mean(vals))
            err.append(np.std(vals, ddof=0) / np.sqrt(len(vals)))
    return np.array(proj), np.array(err)


# ======================================================
# CMS Style Plotting
# ======================================================
def _plot_cms_style(centers, data, data_err, model, xlabel, title, y_label="Efficiency", ylim=None, path_dir="plots"):
    fig = plt.figure(figsize=(8, 8))
    gs = gridspec.GridSpec(2, 1, height_ratios=[3.5, 1.5], hspace=0.05)
    ax0 = plt.subplot(gs[0])
    ax1 = plt.subplot(gs[1], sharex=ax0)

    # --- AUTO-SCALING LOGIC ---
    if ylim is None:
        # Calculamos el punto más alto considerando el error superior
        # Usamos nanmax para ignorar posibles NaNs
        max_data = np.nanmax(data + data_err) if data_err is not None else np.nanmax(data)
        max_model = np.nanmax(model)
        global_max = max(max_data, max_model)
        
        # Si por alguna razón todo es 0 o NaN, ponemos un default
        if np.isnan(global_max) or global_max <= 0:
            global_max = 1.0
            
        # Definimos el límite: 0 abajo, y Max + 30% arriba para la leyenda/CMS label
        current_ylim = (0.0, global_max * 1.30)
    else:
        current_ylim = ylim
    # --------------------------

    # Plot principal
    ax0.plot(centers, model, '-', color='blue', linewidth=2.5, label="Bernstein Model")
    ax0.errorbar(centers, data, yerr=data_err, fmt='ks', markersize=5, elinewidth=1.5, capsize=2, label="Binned MC")
    ax0.set_ylabel(y_label, fontsize=16)
    ax0.set_title(title, loc='center', fontsize=14, fontweight='medium', y=1.05)
    
    # Aplicamos el límite calculado o el manual
    ax0.set_ylim(current_ylim)

    hep.cms.label(data=False, loc=0, ax=ax0, rlabel="13 TeV", fontname="sans-serif", fontsize=16)
    ax0.legend(frameon=False, fontsize=13, loc='upper right')
    ax0.grid(True, alpha=0.3)
    plt.setp(ax0.get_xticklabels(), visible=False)

    # Pulls (El resto sigue igual)
    with np.errstate(divide='ignore', invalid='ignore'):
        pulls = (data - model) / data_err
    pulls[~np.isfinite(pulls)] = 0.0 

    width = centers[1] - centers[0]
    lower = centers[0] - width/2
    upper = centers[-1] + width/2

    ax1.errorbar(centers, pulls, yerr=1.0, xerr=0, fmt='ks',markersize=4, elinewidth=1.0,capsize=0)          
    ax1.axhline(0, color='black', linewidth=1.0, linestyle='-')
    ax1.axhline(3, color='gray', linestyle=':', linewidth=1, alpha=0.8) 
    ax1.axhline(-3, color='gray', linestyle=':', linewidth=1, alpha=0.8)    
    ax1.fill_between([lower, upper], -3, 3, color='gray', alpha=0.15, label=r'$3\sigma$') 
    ax1.set_xlabel(xlabel, fontsize=16)
    ax1.set_ylabel(r'Pull $(\sigma)$', fontsize=13)
    ax1.set_xlim(lower, upper)
    ax1.set_ylim(-4.9, 4.9)
    ax1.grid(True, alpha=0.3)
    ax0.tick_params(axis='both', which='major', labelsize=14, direction='in', top=True, right=True) 
    ax1.tick_params(axis='both', which='major', labelsize=14, direction='in', top=True, right=True)
    
    # Guardado seguro
    save_path = os.path.join(path_dir, f"{title}.png")
    directory = os.path.dirname(save_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ======================================================
# Public Functions (Wrappers)
# ======================================================
def plot_projection_x_with_errors(xc, data2d, model2d, mask, title, ylim=None, path=None):
    """
    Proyección en X (cosThetaL)
    """

    data_proj, data_err = project_with_errors_x(data2d, mask)
    model_proj, _ = project_with_errors_x(model2d, mask)
    if np.nanmean(model_proj) != 0:
        scale = np.nanmean(data_proj) / np.nanmean(model_proj)
    else:
        scale = 1.0
    model_proj_scaled = model_proj * scale
    _plot_cms_style(centers=xc, data=data_proj, data_err=data_err, model=model_proj_scaled, xlabel=r"$\cos\theta_\ell$", title=title, ylim=ylim, path_dir=path)


def plot_projection_y_with_errors(yc, data2d, model2d, mask, title, ylim, path):
    """
    Proyección en Y (cosThetaK)
    """
    data_proj, data_err = project_with_errors_y(data2d, mask)
    model_proj, _ = project_with_errors_y(model2d, mask)
    
    if np.nanmean(model_proj) != 0:
        scale = np.nanmean(data_proj) / np.nanmean(model_proj)
    else:
        scale = 1.0
    model_proj_scaled = model_proj * scale

    _plot_cms_style(centers=yc, data=data_proj, data_err=data_err, model=model_proj_scaled, xlabel=r"$\cos\theta_K$", title=title, ylim=ylim, path_dir=path) 


def select_q2_bin(df, n_bin, cut):
    q2_bins = dict()
    q2_bins = { "bin0":[1.1,23.0],   "bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0],
                "bin4":[6.0, 7.0],   "bin5":[7.0, 8.0], "bin6": [8.0, 11.0],"bin7":[11.0, 12.5],
                "bin8":[12.5, 15.0], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
    df_ = df[(df[cut]>=q2_bins[n_bin][0]) & (df[cut] <= q2_bins[n_bin][1])].copy()
    return df_



# ======================================================
# PHI  1D Bernstei
# ======================================================

def save_bernstein1d_model(filename, coef, n):
    directory = os.path.dirname(filename)
    if directory:
        os.makedirs(directory, exist_ok=True)
    model = {"n": n, "coef": coef.tolist(), "range": [-np.pi, np.pi]}
    with open(filename, "w") as f:
        json.dump(model, f, indent=2)

def bernstein1d_matrix(n, x):
    """
    Build the design matrix for Bernstein 1D basis.
    Input: x in [-1, 1] (array)
    Output: B matrix of size (Npoints, n+1)
    """
    # map [-1,1] -> [0,1]
    t = 0.5 * (x + 1.0)
    B_list = []
    for i in range(n+1):
        B_list.append(bernstein_1d(n, i, t))
    B = np.vstack(B_list).T

    return B


def fit_bernstein1d(xcenters, eff1d, ngen1d, n=4, min_counts_mask=None, reg_lambda=1e-10):
    """
    Fits a Bernstein 1D polynomial to a 1D efficiency histogram.
    """

    if min_counts_mask is None:
        use = (~np.isnan(eff1d)) & (ngen1d > 0)
    else:
        use = min_counts_mask & ~np.isnan(eff1d) & (ngen1d > 0)
        
    x_use = xcenters[use]
    eff_use = eff1d[use]
    ngen_use = ngen1d[use]
    # Binomial uncertainty weights
    sigma2 = eff_use * (1.0 - eff_use) / ngen_use
    sigma2 = np.clip(sigma2, 1e-12, None)
    w = 1.0 / np.sqrt(sigma2)
    
    B = bernstein1d_matrix(n, x_use)
    Bw = B * w[:, None]
    yw = eff_use * w
    BTB = Bw.T @ Bw + reg_lambda * np.eye(B.shape[1])
    BTy = Bw.T @ yw
    coef = np.linalg.solve(BTB, BTy)
    Bfull = bernstein1d_matrix(n, xcenters)
    eff_model = Bfull @ coef
    
    return coef, eff_model


def load_bernstein1d_model(filename):
    """Carga un modelo de Bernstein 1D desde un JSON."""

    with open(filename, 'r') as f:
        data = json.load(f)
    coef = np.asarray(data["coef"], dtype=np.float64)
    if "degree" in data:
        degree = data["degree"]
    else:
        degree = len(coef) - 1
        
    return coef, degree


def plot_1d_result(centers, data, model, mask, title, ylim, path):
    valid = mask
    dummy_err = np.zeros_like(data)
    dummy_err[valid] = 0.05 * data[valid]     
    _plot_cms_style(centers[valid], data[valid], dummy_err[valid], model[valid], xlabel=r"$\phi$", title=title,ylim=ylim, path_dir=path)


def run_fit(model, data):
    nll = zfit.loss.UnbinnedNLL(model=model, data=data)
    #minimizer = zfit.minimize.Minuit()
    minimizer = zfit.minimize.Minuit(maxiter=10000, tol=1e-3)
    result = minimizer.minimize(nll)
    err = None
    try:
        err, _ = result.errors(name="minos", method="minuit_minos", cl=0.682)
    except Exception as e:
        print("MINOS failed:", e)
    return result, err

# =====================================================
# CODE FOR FIT INCLUDING EFFICIENCY
# =====================================================

def tf_bernstein_basis_vectorized(n, t):
    M = tf.shape(t)[0]
    k = tf.range(n + 1, dtype=tf.float64)
    n_float = tf.cast(n, tf.float64)
    log_binom = tf.math.lgamma(n_float + 1.0) - tf.math.lgamma(k + 1.0) - tf.math.lgamma(n_float - k + 1.0)
    binom = tf.exp(log_binom)
    t_col = tf.expand_dims(t, -1) 
    k_row = tf.expand_dims(k, 0)
    term1 = tf.pow(t_col, k_row)
    term2 = tf.pow(1.0 - t_col, n_float - k_row)
    basis = binom * term1 * term2 
    return basis


class Efficiency_Bernstein_Factorized(zfit.pdf.BasePDF):
    def __init__(self, obs,coef_acc_2d, coef_acc_phi, nx_acc, ny_acc, n_phi_acc, coef_reco_2d, coef_reco_phi, nx_reco, ny_reco, n_phi_reco,
                 name="Full_Efficiency_Model"):
        """
        Modelo Completo: Aceptancia * Eficiencia de Reconstrucción.
        Cada parte factorizada en 2D(cosL, cosK) * 1D(phi).
        """
        params = {
            'c_acc_2d': zfit.Parameter(f"c_a2d_{name}", tf.cast(coef_acc_2d, tf.float64), floating=False),
            'c_acc_phi': zfit.Parameter(f"c_aphi_{name}", tf.cast(coef_acc_phi, tf.float64), floating=False),
            'c_reco_2d': zfit.Parameter(f"c_r2d_{name}", tf.cast(coef_reco_2d, tf.float64), floating=False),
            'c_reco_phi': zfit.Parameter(f"c_rphi_{name}", tf.cast(coef_reco_phi, tf.float64), floating=False),
        }
        
        # grados de los polinomios
        self.nx_acc, self.ny_acc = nx_acc, ny_acc
        self.n_phi_acc = n_phi_acc
        self.nx_reco, self.ny_reco = nx_reco, ny_reco
        self.n_phi_reco = n_phi_reco
        
        super().__init__(obs, params, name=name)

    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l, cos_k, phi = vars_list[0], vars_list[1], vars_list[2]

        # [-1, 1] -> [0, 1] y [-pi, pi] -> [0, 1]
        tx = 0.5 * (cos_l + 1.0)
        ty = 0.5 * (cos_k + 1.0)
        t_phi = (phi + np.pi) / (2.0 * np.pi)
        # ======================================================
        # ACEPTANCIA
        # ======================================================
        # Bases
        Bx_acc = tf_bernstein_basis_vectorized(self.nx_acc, tx)
        By_acc = tf_bernstein_basis_vectorized(self.ny_acc, ty)
        Bphi_acc = tf_bernstein_basis_vectorized(self.n_phi_acc, t_phi)
        
        # 2D part
        c_acc_2d_mat = tf.reshape(self.params['c_acc_2d'], (self.nx_acc + 1, self.ny_acc + 1))
        acc_2d = tf.einsum('mi,mj,ij->m', Bx_acc, By_acc, c_acc_2d_mat)
        # 1D part
        acc_phi = tf.einsum('mk,k->m', Bphi_acc, self.params['c_acc_phi'])
        # Total Acceptance
        total_acc = acc_2d * acc_phi

        # ======================================================
        # EFICIENCIA DE RECONSTRUCCIÓN
        # ======================================================
        # Bases
        Bx_reco = tf_bernstein_basis_vectorized(self.nx_reco, tx)
        By_reco = tf_bernstein_basis_vectorized(self.ny_reco, ty)
        Bphi_reco = tf_bernstein_basis_vectorized(self.n_phi_reco, t_phi)
        
        # 2D part
        c_reco_2d_mat = tf.reshape(self.params['c_reco_2d'], (self.nx_reco + 1, self.ny_reco + 1))
        reco_2d = tf.einsum('mi,mj,ij->m', Bx_reco, By_reco, c_reco_2d_mat)
        # 1D part
        reco_phi = tf.einsum('mk,k->m', Bphi_reco, self.params['c_reco_phi'])
        # Total Reco Efficiency
        total_reco = reco_2d * reco_phi

        # ======================================================
        # EFICIENCIA FINAL
        # ======================================================
        return tf.maximum(total_acc * total_reco, 1e-15)


def save_fit_results(result, bin_n, base_dir="fit_results", name="fit_results"):

    
    output_folder = os.path.join(base_dir, f"{bin_n}")
    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{name}.json")
    
    params_dict = {}
    
    for p in result.params:
        val = result.params[p]['value']
        p_data = result.params[p] 

        lower_err = 0.0
        upper_err = 0.0
        sym_err = 0.0
        error_type = "none"

        # busca minos
        if 'minos' in p_data:
            err_data = p_data['minos']
            lower_err = err_data.get('lower', 0.0)
            upper_err = err_data.get('upper', 0.0)
            sym_err = (abs(lower_err) + abs(upper_err)) / 2.0
            error_type = "minos (custom)"

        # busca minuit_minos
        elif 'minuit_minos' in p_data:
            err_data = p_data['minuit_minos']
            lower_err = err_data.get('lower', 0.0)
            upper_err = err_data.get('upper', 0.0)
            sym_err = (abs(lower_err) + abs(upper_err)) / 2.0
            error_type = "minos (default)"
            
        # busca Hesse
        elif 'minuit_hesse' in p_data:
            err_data = p_data['minuit_hesse']
            sym_err = err_data.get('error', -999.0)
            lower_err = -sym_err
            upper_err = sym_err
            error_type = "hesse"
            

        params_dict[p.name] = {'value': float(val), 'error': float(sym_err), 'error_low': float(lower_err), 'error_up': float(upper_err), 'error_source': error_type}


    cov_matrix = result.covariance()
    cov_list = np.array(cov_matrix).tolist()
    data_to_save = {'bin_index': str(bin_n), 'valid': bool(result.valid), 'converged': bool(result.converged), 'fmin': float(result.fmin), 'status': result.status,'parameters': params_dict,'covariance': cov_list}
    
    with open(output_file, 'w') as f:
        json.dump(data_to_save, f, indent=4)
        
    print(f"[CheckPoint] Resultados guardados en: {output_file}")
    return output_file


def save_correlation_matrix(result, params_list, bin_name, out_dir="plots/correlations"):

    os.makedirs(out_dir, exist_ok=True)    
    corr_matrix_raw = result.correlation()    
    zfit_params = list(result.params.keys())
    n_params = len(params_list)
    corr_matrix = np.zeros((n_params, n_params))    
    param_names = [p.name.split('_')[0] for p in params_list]    
    for i, p1 in enumerate(params_list):
        for j, p2 in enumerate(params_list):
            idx1 = zfit_params.index(p1)
            idx2 = zfit_params.index(p2)
            corr_matrix[i, j] = corr_matrix_raw[idx1, idx2]
                
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, xticklabels=param_names, yticklabels=param_names, fmt=".2f", square=True, cbar_kws={"shrink": .8})
    plt.title(f"Matriz de Correlación - {bin_name}", fontsize=14, pad=15)
    plt.xticks(rotation=45)
    plt.tight_layout()
    filepath = os.path.join(out_dir, f"corr_matrix_{bin_name}.png")
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close() 

    


#  CODE FOR EFFICIENCY CALCULATION

In [3]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")



1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [4]:

recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=22)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=2)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.3, random_state=2)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=2)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

In [5]:
print("Numero de eventes GEN",len(obs_Gen))
print("Numero de eventes GEN Filtered",len(obs_GenFtr))
print("Numero de eventes Reco Gen filtered",len(obs_RecoGenFtr))
print("Numero de eventes Reco Filtered",len(obs_RecoFtr))


Numero de eventes GEN 347675
Numero de eventes GEN Filtered 30769
Numero de eventes Reco Gen filtered 1889406
Numero de eventes Reco Filtered 90043


# NUEVAS FUNVIONES Y PDF

In [6]:
import zfit
from zfit import z
import tensorflow as tf
import numpy as np
import pandas as pd

# PDF física completa
class FullAngular_Physical_PDF(zfit.pdf.BasePDF):
    def __init__(self, obs, FL, S3, S9, AFB, S4, S7, S5, S8, name="FullAngular_Physical_PDF"):
        params = {
            'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB,
            'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8
        }
        super().__init__(obs, params, name=name)
    
    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l = vars_list[0]
        cos_k = vars_list[1]
        phi   = vars_list[2]
        sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 0.0))
        sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 0.0))
        sin2_k = sin_k**2
        cos2_k = cos_k**2
        sin2_l = sin_l**2
        
        cos2l_term = 2.0 * cos_l**2 - 1.0
        sin2l_term = 2.0 * sin_l * cos_l
        sin2k_term = 2.0 * sin_k * cos_k
        
        cos_phi = tf.cos(phi)
        sin_phi = tf.sin(phi)
        cos2_phi = tf.cos(2.0 * phi)
        sin2_phi = tf.sin(2.0 * phi)

        FL = self.params['FL']
        S3 = self.params['S3']
        S9 = self.params['S9']
        AFB = self.params['AFB']
        S4 = self.params['S4']
        S7 = self.params['S7']
        S5 = self.params['S5']
        S8 = self.params['S8']
        
        term1 = 0.75 * (1.0 - FL) * sin2_k
        term2 = FL * cos2_k
        term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
        term4 = -1.0 * FL * cos2_k * cos2l_term
        term5 = S3 * sin2_k * sin2_l * cos2_phi
        term6 = S4 * sin2k_term * sin2l_term * cos_phi
        term7 = S5 * sin2k_term * sin_l * cos_phi
        term8 = (4.0/3.0) * AFB * sin2_k * cos_l
        term9 = S7 * sin2k_term * sin_l * sin_phi
        term10 = S8 * sin2k_term * sin2l_term * sin_phi
        term11 = S9 * sin2_k * sin2_l * sin2_phi
        
        pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
        return pdf

    @zfit.supports(norm=True)
    def _integrate(self, limits, norm_range, options=None):
        integral_value = 32.0 * np.pi / 9.0
        return tf.constant([integral_value], dtype=self.dtype)


# PDF transformada usando el método Polar 
class FullAngular_Transformed_PDF(zfit.pdf.BasePDF):
    def __init__(self, obs, raw_FL, raw_S3, raw_r1, alpha1, raw_r2, alpha2, raw_r3, alpha3, name="FullAngular_Transformed_PDF"):
        params = {
            'rFL': raw_FL, 'rS3': raw_S3, 
            'r1': raw_r1, 'a1': alpha1,
            'r2': raw_r2, 'a2': alpha2,
            'r3': raw_r3, 'a3': alpha3
        }
        super().__init__(obs, params, name=name)

    def _get_physical_coeffs(self):
        # FL y S3
        FL = 0.5 * (1.0 + tf.math.tanh(self.params['rFL']))
        FT = 1.0 - FL
        S3 = 0.5 * FT * tf.math.tanh(self.params['rS3'])
    
        # (S9, AFB)
        R1_sq = tf.maximum(0.25 * tf.square(FT) - tf.square(S3), 1e-18)
        R1 = tf.sqrt(R1_sq)
        r1_phys = (R1 / 2.0) * (1.0 + tf.math.tanh(self.params['r1']))
        a1 = self.params['a1']
    
        AFB = 1.5 * tf.math.cos(a1) * r1_phys
        S9 = tf.math.sin(a1) * r1_phys
    
        # (S4, S7)
        R2_sq = tf.maximum(FL * (FT - 2.0 * S3), 1e-18)
        R2 = tf.sqrt(R2_sq)
        r2_phys = (R2 / 2.0) * (1.0 + tf.math.tanh(self.params['r2']))
        a2 = self.params['a2']
    
        S4 = 0.5 * tf.math.cos(a2) * r2_phys
        S7 = tf.math.sin(a2) * r2_phys
    
        # (S5, S8)
        R3_sq = tf.maximum(FL * (FT + 2.0 * S3), 1e-18)
        R3 = tf.sqrt(R3_sq)
        r3_phys = (R3 / 2.0) * (1.0 + tf.math.tanh(self.params['r3']))
        a3 = self.params['a3']
    
        S5 = tf.math.cos(a3) * r3_phys
        S8 = 0.5 * tf.math.sin(a3) * r3_phys
    
        return FL, S3, S9, AFB, S4, S7, S5, S8

    def _unnormalized_pdf(self, x):
        vars_list = z.unstack_x(x)
        cos_l = vars_list[0]
        cos_k = vars_list[1]
        phi   = vars_list[2]
    
        # sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 0.0), 1e-10)
        # sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 0.0),1e-10)     
        sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 1e-10))
        sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 1e-10))   
        sin2_k = sin_k**2
        cos2_k = cos_k**2
        sin2_l = sin_l**2
        cos2l_term = 2.0 * cos_l**2 - 1.0
        sin2l_term = 2.0 * sin_l * cos_l
        sin2k_term = 2.0 * sin_k * cos_k
        cos_phi = tf.cos(phi)
        sin_phi = tf.sin(phi)
        cos2_phi = tf.cos(2.0 * phi)
        sin2_phi = tf.sin(2.0 * phi)

        FL, S3, S9, AFB, S4, S7, S5, S8 = self._get_physical_coeffs()
    
        term1 = 0.75 * (1.0 - FL) * sin2_k
        term2 = FL * cos2_k
        term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
        term4 = -1.0 * FL * cos2_k * cos2l_term
        term5 = S3 * sin2_k * sin2_l * cos2_phi
        term6 = S4 * sin2k_term * sin2l_term * cos_phi
        term7 = S5 * sin2k_term * sin_l * cos_phi
        term8 = (4.0/3.0) * AFB * sin2_k * cos_l
        term9 = S7 * sin2k_term * sin_l * sin_phi
        term10 = S8 * sin2k_term * sin2l_term * sin_phi
        term11 = S9 * sin2_k * sin2_l * sin2_phi
    
        pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
        return pdf

    @zfit.supports(norm=True)
    def _integrate(self, limits, norm_range, options=None):
        integral_value = 32.0 * np.pi / 9.0
        return tf.constant([integral_value], dtype=self.dtype)


def get_inverse_values(phys_params):
    """
    Toma los valores físicos (FL, S3, S9, AFB, S4, S7, S5, S8) y devuelve los 
    parámetros iniciales sin restricciones (rFL, rS3, r1, a1, r2, a2, r3, a3).
    """
    FL, S3, S9, AFB, S4, S7, S5, S8 = phys_params
    def atanh(x): return np.arctanh(np.clip(x, -0.9999, 0.9999))

    rFL = atanh(2.0 * FL - 1.0)
    FT = 1.0 - FL
    rS3 = atanh(S3 / (0.5 * FT))

    # Inversa Sector 1 (Transversal)
    x1 = (2.0/3.0) * AFB
    y1 = S9
    r1_phys = np.sqrt(x1**2 + y1**2)
    a1 = np.arctan2(y1, x1)

    R1 = np.sqrt(max(1e-15, 0.25 * FT**2 - S3**2))
    ratio1 = r1_phys / R1 if R1 > 0 else 0
    r1 = atanh(2.0 * ratio1 - 1.0)

    # Inversa Sector 2 (Paralelo)
    x2 = 2.0 * S4
    y2 = S7
    r2_phys = np.sqrt(x2**2 + y2**2)
    a2 = np.arctan2(y2, x2)

    R2 = np.sqrt(max(1e-15, FL * (FT - 2.0 * S3)))
    ratio2 = r2_phys / R2 if R2 > 0 else 0
    r2 = atanh(2.0 * ratio2 - 1.0)

    # Inversa Sector 3 (Perpendicular)
    x3 = S5
    y3 = 2.0 * S8
    r3_phys = np.sqrt(x3**2 + y3**2)
    a3 = np.arctan2(y3, x3)

    R3 = np.sqrt(max(1e-15, FL * (FT + 2.0 * S3)))
    ratio3 = r3_phys / R3 if R3 > 0 else 0
    r3 = atanh(2.0 * ratio3 - 1.0)

    return [rFL, rS3, r1, a1, r2, a2, r3, a3]


def apply_transformation_equations(rFL, rS3, r1, a1, r2, a2, r3, a3):
    """
    Aplica las ecuaciones de transformación del espacio polar TRANSFORMADO al FISICO.
    """
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)

    # Sector 1
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    r1_phys = (R1 / 2.0) * (1.0 + np.tanh(r1))
    AFB = 1.5 * np.cos(a1) * r1_phys
    S9 = np.sin(a1) * r1_phys

    # Sector 2
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    r2_phys = (R2 / 2.0) * (1.0 + np.tanh(r2))
    S4 = 0.5 * np.cos(a2) * r2_phys
    S7 = np.sin(a2) * r2_phys

    # Sector 3
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    r3_phys = (R3 / 2.0) * (1.0 + np.tanh(r3))
    S5 = np.cos(a3) * r3_phys
    S8 = 0.5 * np.sin(a3) * r3_phys

    return {'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB, 'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8}


# PDF transformada usando el método Cartesiano No Acotado (Sin degeneraciones)
# class FullAngular_Transformed_PDF(zfit.pdf.BasePDF):
#     def __init__(self, obs, raw_FL, raw_S3, u1, v1, u2, v2, u3, v3, name="FullAngular_Transformed_PDF"):
#         params = {
#             'rFL': raw_FL, 'rS3': raw_S3, 
#             'u1': u1, 'v1': v1,
#             'u2': u2, 'v2': v2,
#             'u3': u3, 'v3': v3
#         }
#         super().__init__(obs, params, name=name)

#     def _cartesian_to_physical(self, u, v, R, scale_x, scale_y):
#         """
#         Mapea los parámetros libres (u, v) al interior del círculo de radio R
#         y escala para obtener los observables físicos.
#         """
#         rho_sq = tf.square(u) + tf.square(v)
#         rho = tf.sqrt(tf.maximum(rho_sq, 1e-18)) # Prevenir NaN en la raíz
        
#         # Aproximación de Taylor para tanh(rho)/rho cerca del origen para estabilidad numérica
#         factor = tf.where(rho_sq < 1e-8, 
#                           1.0 - rho_sq / 3.0, 
#                           tf.math.tanh(rho) / rho)
        
#         x = R * factor * u
#         y = R * factor * v
        
#         return x * scale_x, y * scale_y

#     def _get_physical_coeffs(self):
#         # Sector Longitudinal / Transversal Base
#         FL = 0.5 * (1.0 + tf.math.tanh(self.params['rFL']))
#         FT = 1.0 - FL
#         S3 = 0.5 * FT * tf.math.tanh(self.params['rS3'])
    
#         # Sector 1: (S9, AFB)
#         R1_sq = tf.maximum(0.25 * tf.square(FT) - tf.square(S3), 1e-18)
#         R1 = tf.sqrt(R1_sq)
#         AFB, S9 = self._cartesian_to_physical(self.params['u1'], self.params['v1'], R1, scale_x=1.5, scale_y=1.0)
    
#         # Sector 2: (S4, S7)
#         R2_sq = tf.maximum(FL * (FT - 2.0 * S3), 1e-18)
#         R2 = tf.sqrt(R2_sq)
#         S4, S7 = self._cartesian_to_physical(self.params['u2'], self.params['v2'], R2, scale_x=0.5, scale_y=1.0)
    
#         # Sector 3: (S5, S8)
#         R3_sq = tf.maximum(FL * (FT + 2.0 * S3), 1e-18)
#         R3 = tf.sqrt(R3_sq)
#         S5, S8 = self._cartesian_to_physical(self.params['u3'], self.params['v3'], R3, scale_x=1.0, scale_y=0.5)
    
#         return FL, S3, S9, AFB, S4, S7, S5, S8

#     def _unnormalized_pdf(self, x):
#         vars_list = z.unstack_x(x)
#         cos_l = vars_list[0]
#         cos_k = vars_list[1]
#         phi   = vars_list[2]
    
#         sin_k = tf.sqrt(tf.maximum(1.0 - cos_k**2, 1e-10))
#         sin_l = tf.sqrt(tf.maximum(1.0 - cos_l**2, 1e-10))   
#         sin2_k = sin_k**2
#         cos2_k = cos_k**2
#         sin2_l = sin_l**2
#         cos2l_term = 2.0 * cos_l**2 - 1.0
#         sin2l_term = 2.0 * sin_l * cos_l
#         sin2k_term = 2.0 * sin_k * cos_k
#         cos_phi = tf.cos(phi)
#         sin_phi = tf.sin(phi)
#         cos2_phi = tf.cos(2.0 * phi)
#         sin2_phi = tf.sin(2.0 * phi)

#         FL, S3, S9, AFB, S4, S7, S5, S8 = self._get_physical_coeffs()
    
#         term1 = 0.75 * (1.0 - FL) * sin2_k
#         term2 = FL * cos2_k
#         term3 = 0.25 * (1.0 - FL) * sin2_k * cos2l_term
#         term4 = -1.0 * FL * cos2_k * cos2l_term
#         term5 = S3 * sin2_k * sin2_l * cos2_phi
#         term6 = S4 * sin2k_term * sin2l_term * cos_phi
#         term7 = S5 * sin2k_term * sin_l * cos_phi
#         term8 = (4.0/3.0) * AFB * sin2_k * cos_l
#         term9 = S7 * sin2k_term * sin_l * sin_phi
#         term10 = S8 * sin2k_term * sin2l_term * sin_phi
#         term11 = S9 * sin2_k * sin2_l * sin2_phi
    
#         pdf = term1 + term2 + term3 + term4 + term5 + term6 + term7 + term8 + term9 + term10 + term11
#         return pdf

#     @zfit.supports(norm=True)
#     def _integrate(self, limits, norm_range, options=None):
#         integral_value = 32.0 * np.pi / 9.0
#         return tf.constant([integral_value], dtype=self.dtype)


# def get_inverse_values(phys_params):
#     """
#     Toma los valores físicos (FL, S3, S9, AFB, S4, S7, S5, S8) y devuelve los 
#     parámetros iniciales sin restricciones (rFL, rS3, u1, v1, u2, v2, u3, v3).
#     """
#     FL, S3, S9, AFB, S4, S7, S5, S8 = phys_params
#     def atanh(x): return np.arctanh(np.clip(x, -0.999999, 0.999999))

#     rFL = atanh(2.0 * FL - 1.0)
#     FT = 1.0 - FL
#     rS3 = atanh(S3 / (0.5 * FT))

#     def _physical_to_cartesian(x, y, R):
#         if R <= 1e-15:
#             return 0.0, 0.0
#         r_phys = np.sqrt(x**2 + y**2)
#         ratio = np.clip(r_phys / R, 0.0, 0.999999)
#         rho = np.arctanh(ratio)
#         if r_phys < 1e-12:
#             return 0.0, 0.0
#         factor = rho / r_phys
#         return x * factor, y * factor

#     # Inversa Sector 1
#     R1 = np.sqrt(max(1e-15, 0.25 * FT**2 - S3**2))
#     u1, v1 = _physical_to_cartesian((2.0/3.0) * AFB, S9, R1)

#     # Inversa Sector 2
#     R2 = np.sqrt(max(1e-15, FL * (FT - 2.0 * S3)))
#     u2, v2 = _physical_to_cartesian(2.0 * S4, S7, R2)

#     # Inversa Sector 3
#     R3 = np.sqrt(max(1e-15, FL * (FT + 2.0 * S3)))
#     u3, v3 = _physical_to_cartesian(S5, 2.0 * S8, R3)

#     return [rFL, rS3, u1, v1, u2, v2, u3, v3]


# def apply_transformation_equations(rFL, rS3, u1, v1, u2, v2, u3, v3):
#     """
#     Aplica las ecuaciones de transformación del espacio cartesiano sin límites AL FISICO.
#     """
#     FL = 0.5 * (1.0 + np.tanh(rFL))
#     FT = 1.0 - FL
#     S3 = 0.5 * FT * np.tanh(rS3)

#     def _cartesian_to_physical_np(u, v, R, scale_x, scale_y):
#         rho = np.sqrt(u**2 + v**2)
#         factor = np.tanh(rho) / rho if rho > 1e-8 else 1.0 - (rho**2)/3.0
#         x = R * factor * u
#         y = R * factor * v
#         return x * scale_x, y * scale_y

#     # Sector 1
#     R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
#     AFB, S9 = _cartesian_to_physical_np(u1, v1, R1, 1.5, 1.0)

#     # Sector 2
#     R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
#     S4, S7 = _cartesian_to_physical_np(u2, v2, R2, 0.5, 1.0)

#     # Sector 3
#     R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
#     S5, S8 = _cartesian_to_physical_np(u3, v3, R3, 1.0, 0.5)

#     return {'FL': FL, 'S3': S3, 'S9': S9, 'AFB': AFB, 'S4': S4, 'S7': S7, 'S5': S5, 'S8': S8}



def save_phy_results(result_zfit, phys_dict, cov_phys, obs_order, bin_n, base_dir="fit_results", name="fit_results_phys"):
    """
    Guarda los resultados de los observables físicos transformados y su matriz 
    de covarianza propagada en el mismo formato JSON que los resultados del fit.
    """
    output_folder = os.path.join(base_dir, f"{bin_n}")
    os.makedirs(output_folder, exist_ok=True)
    output_file = os.path.join(output_folder, f"{name}.json")
    
    params_dict = {}
    
    errors_phys = np.sqrt(np.diag(cov_phys))
    
    for i, obs_name in enumerate(obs_order):
        val = phys_dict.get(obs_name, 0.0)
        sym_err = errors_phys[i]
        
        # asume errores simétricos NLL "normal"
        params_dict[obs_name] = {
            'value': float(val), 
            'error': float(sym_err), 
            'error_low': float(-sym_err), 
            'error_up': float(sym_err), 
            'error_source': "delta_method"
        }

    cov_list = cov_phys.tolist()
    data_to_save = {
        'bin_index': str(bin_n), 
        'valid': bool(result_zfit.valid), 
        'converged': bool(result_zfit.converged), 
        'fmin': float(result_zfit.fmin), 
        'status': result_zfit.status,
        'parameters': params_dict,
        'covariance': cov_list
    }
    
    with open(output_file, 'w') as f:
        json.dump(data_to_save, f, indent=4)
        
    print(f"[CheckPoint] Resultados físicos (transformados) guardados en: {output_file}")
    return output_file
    

     
def plot_nll_profiles(result, params_list, bin_name, base_dir="plots/profiles"):
    bin_dir = os.path.join(base_dir, bin_name)
    os.makedirs(bin_dir, exist_ok=True)
    
    loss = result.loss
    nll_min = result.fmin
    minimizer = zfit.minimize.Minuit(verbosity=0)
    best_fit_values = {p: result.params[p]['value'] for p in params_list}
    
    for param in params_list:
        clean_name = param.name.split('_')[0]
        print(f"--- Calculando perfil NLL para {param.name} ---")
        
        val_opt = best_fit_values[param]
        minos_data = result.params[param].get('minos', {})
        err_lower = minos_data.get('lower', -0.1)
        err_upper = minos_data.get('upper', 0.1)
        
        scan_min = val_opt + 1.5*err_lower
        scan_max = val_opt + 1.5*err_upper
        
        if param.has_limits:
            p_lower = float(param.lower)
            p_upper = float(param.upper)
            epsilon = 1e-4
            
            scan_min = max(scan_min, p_lower + epsilon)
            scan_max = min(scan_max, p_upper - epsilon)
            
        scan_values = np.linspace(scan_min, scan_max, 50)
        nll_values = []
        
        # Escaneo de NLL
        for val_scan in scan_values:
            for p in params_list:
                p.set_value(best_fit_values[p])
            
            param.set_value(val_scan)   
            param.floating = False
            temp_result = minimizer.minimize(loss)
            nll_values.append(temp_result.fmin)


                
        param.floating = True
        nll_values = np.array(nll_values)
        delta_nll = nll_values - nll_min
        
        # ==========================================
        # plots
        # ==========================================
        fig, ax = plt.subplots(figsize=(8, 6))

        ax.plot(scan_values, delta_nll, '-', lw=1, color='blue', label='Profile Likelihood')
        ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label=r'$\Delta$NLL = 0.5 (Errors $1\sigma$)')
        ax.axvline(val_opt, color='green', linestyle='-.', linewidth=1.5, label='Best fit')   

        ax.set_title(f'NLL Profile: {clean_name} ({bin_name})', loc='center', fontsize=14, fontweight='medium', y=1.05)
        ax.set_xlabel(f'{clean_name}', fontsize=16)
        ax.set_ylabel(r'$\Delta$ NLL', fontsize=16)
        ax.set_ylim(bottom=-0.1, top=0.52) 

        hep.cms.label(data=False, loc=0, ax=ax, rlabel="13 TeV", fontname="sans-serif", fontsize=16)

        ax.legend(frameon=False, fontsize=13, loc='upper right') # 'upper center' también suele funcionar bien en perfiles NLL
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='both', which='major', labelsize=14, direction='in', top=True, right=True)

        file_path = f"{bin_dir}/nll_profile_{clean_name}_{bin_name}.png"
        plt.savefig(file_path, dpi=300, bbox_inches='tight')
        plt.close()
        # plt.figure(figsize=(8, 6))
        # plt.plot(scan_values, delta_nll, '-', lw=1.5, label='Perfil Likelihood')
        # plt.axhline(0.5, color='r', linestyle='--', label=r'$\Delta$NLL = 0.5 (Errores a $1\sigma$)')
        # plt.axvline(val_opt, color='g', linestyle='-.', label='Mejor ajuste')   
        # plt.title(f'NLL Profile: {clean_name} ({bin_name})', fontsize=14)
        # plt.xlabel(f'{clean_name}', fontsize=12)
        # plt.ylabel(r'$\Delta$ NLL', fontsize=12)
        # plt.ylim(bottom=-0.1, top=0.52) 
        # plt.legend()
        # plt.grid(True, alpha=0.3)
        # plt.tight_layout()
        # file_path = f"{bin_dir}/nll_profile_{clean_name}_{bin_name}.png"
        # plt.savefig(file_path, dpi=300)
        # plt.close()



def print_polar_physical_variables(rFL, rS3, r1, a1, r2, a2, r3, a3, bin_name=""):
    """
    Calcula e imprime los radios máximos, radios físicos y ángulos 
    reales a partir de los parámetros libres del ajuste.
    """
    #  FL y S3 físicos
    FL = 0.5 * (1.0 + np.tanh(rFL))
    FT = 1.0 - FL
    S3 = 0.5 * FT * np.tanh(rS3)
    
    # calcular los radios
    R1 = np.sqrt(np.maximum(0.25 * FT**2 - S3**2, 1e-15))
    r1_phys = (R1 / 2.0) * (1.0 + np.tanh(r1))
    
    R2 = np.sqrt(np.maximum(FL * (FT - 2.0 * S3), 1e-15))
    r2_phys = (R2 / 2.0) * (1.0 + np.tanh(r2))
    
    R3 = np.sqrt(np.maximum(FL * (FT + 2.0 * S3), 1e-15))
    r3_phys = (R3 / 2.0) * (1.0 + np.tanh(r3))
    
    print("\n" + "="*60)
    print(f"VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - {bin_name}")
    print("="*60)
    
    print(f"[Sector 1: S9, AFB]")
    print(f"  Radio máximo (R1)     = {R1:.6f}")
    print(f"  Radio físico (r1_phys)= {r1_phys:.6f}")
    print(f"  Ángulo (alpha_1)      = {a1:.6f} rad")
    
    print(f"\n[Sector 2: S4, S7]")
    print(f"  Radio máximo (R2)     = {R2:.6f}")
    print(f"  Radio físico (r2_phys)= {r2_phys:.6f}")
    print(f"  Ángulo (alpha_2)      = {a2:.6f} rad")
    
    print(f"\n[Sector 3: S5, S8]")
    print(f"  Radio máximo (R3)     = {R3:.6f}")
    print(f"  Radio físico (r3_phys)= {r3_phys:.6f}")
    print(f"  Ángulo (alpha_3)      = {a3:.6f} rad")
    print("="*60 + "\n")

# Gen  FIT V2 trasnsformation

In [7]:
#%%capture data_gen_fit_transformed_polar
q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
#q2_bins = {"bin9":[15.0, 17.0]}

def get_physical_array(r_array):
    phys_dict = apply_transformation_equations(*r_array)
    return np.array([phys_dict.get(key, 0.0) for key in obs_order])

def compute_jacobian(func, x, epsilon=1e-5):
    n_in = len(x)
    y_center = func(x)
    n_out = len(y_center)
    
    J = np.zeros((n_out, n_in))
    
    for i in range(n_in):
        x_plus = np.copy(x)
        x_minus = np.copy(x)
        
        x_plus[i] += epsilon
        x_minus[i] -= epsilon
        
        y_plus = func(x_plus)
        y_minus = func(x_minus)
        
        J[:, i] = (y_plus - y_minus) / (2.0 * epsilon)
        
    return J


for binN in q2_bins.keys():
    print(f"\n{'='*60}\nProcesando {binN} con rango q2: {q2_bins[binN]}\n{'='*60}")
    json_path = f"fit_results/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
    
    init_rFL, init_rS3 = 0.235, -0.167
    init_r1, init_a1 = -2.210, -1.585
    init_r2, init_a2 = 0.278, 1
    init_r3, init_a3 = -1.160, -1.310

    if os.path.exists(json_path):
        print(f"-> Leyendo semillas iniciales de: {json_path}")
        with open(json_path, 'r') as file:
            data = json.load(file)
        params_dict = data.get("parameters", {})
        
        phys_params_list = [
            params_dict["FL"]["value"],
            params_dict["S3"]["value"],
            params_dict["S9"]["value"],
            params_dict["AFB"]["value"],
            params_dict["S4"]["value"],
            params_dict["S7"]["value"],
            params_dict["S5"]["value"],
            params_dict["S8"]["value"]
        ]
        
        inv_vals = get_inverse_values(phys_params_list)
        print(inv_vals)
        init_rFL, init_rS3, init_r1, init_a1, init_r2, init_a2, init_r3, init_a3 = inv_vals
        

    else:
        print(f"-> ADVERTENCIA: No se encontró el archivo {json_path}")


    # ======================================================
    # CONFIGURACIÓN DEL ESPACIO 
    # ======================================================
    cos_l = zfit.Space('cos_l', limits=(-1, 1))
    cos_k = zfit.Space('cos_k', limits=(-1, 1))
    phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
    obs_ang = cos_l * cos_k * phi  

    # ======================================================
    # CREACIÓN DE PARÁMETROS ZFIT CON LAS SEMILLAS DINÁMICAS
    # ======================================================
    limit =3
    # F_L y S3
    rFL = zfit.Parameter(f'rFL_{binN}', init_rFL, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    rS3 = zfit.Parameter(f'rS3_{binN}', init_rS3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    
    # S9y AFB 
    r1 = zfit.Parameter(f'rr1_{binN}', init_r1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    a1 = zfit.Parameter(f'a1_{binN}', init_a1, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    
    # S4 y S7 
    r2 = zfit.Parameter(f'rr2_{binN}', init_r2, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    a2 = zfit.Parameter(f'a2_{binN}', init_a2, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    
    # 3 S5 y S8 
    r3 = zfit.Parameter(f'rr3_{binN}', init_r3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)
    a3 = zfit.Parameter(f'a3_{binN}', init_a3, step_size=1e-3, lower_limit=-limit, upper_limit=limit)

    fit_params_list = [rFL, rS3, r1, a1, r2, a2, r3, a3]


    # ======================================================
    # PDFs Y CARGA DE DATOS
    # ======================================================
    
    pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, r1, a1, r2, a2, r3, a3)
    obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
    print("NÚMERO DE EVENTOS:", len(obs_Gen_q2),"\n")
    data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)

    # ======================================================
    # FITS
    # ======================================================

    print("\n" + "="*60)
    print(">>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<")
    print("="*60)
    result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
    save_fit_results(result_gen, binN, base_dir="fit_results/gen_transformed_polar", name=f"fit_results_gen_physical_{binN}")
    r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
    cov_fit = result_gen.covariance(params=fit_params_list)
    print(result_gen.params)
    print_polar_physical_variables(*r_values, bin_name=binN)
    obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
    J = compute_jacobian(get_physical_array, r_values)
    cov_phys = J @ cov_fit @ J.T
    errors_phys = np.sqrt(np.diag(cov_phys))

    phys_vals_gen_dict = apply_transformation_equations(*r_values)
    save_phy_results(result_zfit=result_gen, phys_dict=phys_vals_gen_dict, cov_phys=cov_phys, obs_order=obs_order, bin_n=binN, base_dir="fit_results/gen_transformed_polar", name=f"fit_results_gen_transformed_phys_{binN}")
    print("\n" + "-"*80)
    print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
    print("-"*80)
    print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
    print("-"*80)

    for i, key in enumerate(obs_order):
        val = phys_vals_gen_dict.get(key, 0.0)
        err = errors_phys[i]
        print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
    print("-"*80 + "\n")
    # plot_nll_profiles(result_gen, fit_params_list, binN)
    # save_correlation_matrix(result_gen, fit_params_list, binN, out_dir=f"fit_results/gen_transformed_polar/{binN}")




Procesando bin1 con rango q2: [1.1, 2.0]
-> Leyendo semillas iniciales de: fit_results/gen_phy_slsqp/bin1/fit_results_gen_physical_slsqp_bin1.json
[0.37139004652924373, 0.006349338387853855, -1.4295955242669218, -1.759523979871487, -0.9602105382991027, 0.21588613993249625, -1.4831868010617832, -2.045824760682309]
NÚMERO DE EVENTOS: 11739 


>>> INICIANDO FIT GEN LEVEL TRANSFORMED SPACE <<<


2026-03-09 23:07:40.816734: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-09 23:07:41.440127: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-09 23:07:41.442432: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

[CheckPoint] Resultados guardados en: fit_results/gen_transformed_polar/bin1/fit_results_gen_physical_bin1.json
name        value  (rounded)                minos    at limit
--------  ------------------  -------------------  ----------
rFL_bin1             0.37139  -  0.014   +  0.014       False
rS3_bin1          0.00634722  -  0.052   +  0.052       False
rr1_bin1             -1.4296  -    1.3   +   0.37       False
a1_bin1             -1.75953  -    1.3   +   0.61       False
rr2_bin1            -0.96021  -   0.23   +   0.17       False
a2_bin1             0.215886  -   0.17   +   0.21       False
rr3_bin1            -1.48318  -    0.7   +   0.33       False
a3_bin1             -2.04583  -    1.1   +   0.45       False

VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - bin1
[Sector 1: S9, AFB]
  Radio máximo (R1)     = 0.161195
  Radio físico (r1_phys)= 0.008738
  Ángulo (alpha_1)      = -1.759525 rad

[Sector 2: S4, S7]
  Radio máximo (R2)     = 0.465908
  Radio físico (r2_phys

2026-03-09 23:08:04.136662: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-09 23:08:04.308917: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-09 23:08:04.319332: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-09 23:08:04.331993: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-09 23:08:04.370837: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-09 23:08:04.384166: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7ebff3c00f00 = {1, 0} Ch

[CheckPoint] Resultados guardados en: fit_results/gen_transformed_polar/bin4/fit_results_gen_physical_bin4.json
name        value  (rounded)                minos    at limit
--------  ------------------  -------------------  ----------
rFL_bin4            0.234814  -  0.012   +  0.012       False
rS3_bin4           -0.167234  -  0.042   +  0.041       False
rr1_bin4            -2.20796  -    1.7   +   0.76       False
a1_bin4             -1.58604  -    1.6   +    1.6       False
rr2_bin4             1.27941  -   0.23   +   0.38       False
a2_bin4              3.11478  -  0.018   +  0.018       False
rr3_bin4            -2.15767  -    1.3   +    0.7       False
a3_bin4             -2.31144  -    2.3   +    2.3       False

VARIABLES POLARES FÍSICAS (Radios y Ángulos Reales) - bin4
[Sector 1: S9, AFB]
  Radio máximo (R1)     = 0.189693
  Radio físico (r1_phys)= 0.002265
  Ángulo (alpha_1)      = -1.586041 rad

[Sector 2: S4, S7]
  Radio máximo (R2)     = 0.525288
  Radio físico (r2_phys

In [8]:
# with open('data_gen_fit_transformed_polar.txt', 'w') as archivo:
#     archivo.write(data_gen_fit_transformed_polar.stdout) 
#     archivo.write(data_gen_fit_transformed_polar.stderr)
